In [2]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()


llm = ChatOpenAI(
    model="gpt-5",
    temperature=0.4,
    max_retries=0,
    api_key=os.getenv("OPENAI_API_KEY"),
)

llm.invoke("Hello, how are you?")

AIMessage(content='Hi there! I’m doing well, thanks for asking. How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 12, 'total_tokens': 104, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-C66YQyfUiIo1qHcy57fWSkV7kD58G', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--48672e7e-1688-410d-8c4d-afd30f03b85a-0', usage_metadata={'input_tokens': 12, 'output_tokens': 92, 'total_tokens': 104, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 64}})

In [ ]:
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph
from pydantic import BaseModel, Field, ValidationError
from typing_extensions import Annotated, Literal


class State(BaseModel):
    route_selector: Annotated[
        Literal["a", "b", ""], Field(default="", description="The route selector")
    ]
    route_log: Annotated[
        list[str], Field(default_factory=list, description="The route log")
    ]
    visit_count: Annotated[int, Field(default=0, description="The visit count")]


def start_node(state: State) -> State:
    state.route_log.append("Start_node")
    return state


def route_select_to_a(state: State) -> State:
    state.route_log.append("Route_select_to_a")
    return state


def route_select_to_b(state: State) -> State:
    if "Route_select_to_b" in state.route_log:
        state.visit_count += 1
        state.route_log.append("Route_select_to_b is already in the route log")
    else:
        state.route_log.append("Route_select_to_b")

    return state


def route_selector(state: State) -> Literal["a", "b"]:
    if state.visit_count < 2:
        state.route_selector = "a"
        return state.route_selector
    else:
        state.route_selector = "b"
        return state.route_selector


def end_node(state: State) -> State:
    state.route_log.append("End_node")
    return state


config = RunnableConfig(
    configurable={"thread_id": "1"},
    tags=["my-rag"],
)


memory = MemorySaver()

workflow = StateGraph(State)

workflow.add_node("start", start_node)
workflow.add_node("route_select_to_a", route_select_to_a)
workflow.add_node("route_select_to_b", route_select_to_b)
workflow.add_node("route_selector", route_selector)
workflow.add_node("end", end_node)

workflow.set_entry_point("start")
workflow.add_edge("start", "route_select_to_a")
workflow.add_edge("route_select_to_a", "route_select_to_b")
workflow.add_edge("route_select_to_b", "end")

graph = workflow.compile(checkpointer=memory)


try:
    graph.invoke(
        State(
            route_selector="",
            route_log=[],
            visit_count=0,
        ),
        config=config,
    )
except ValidationError as e:
    print(e.errors()[0])

graph.get_state(config=config)

StateSnapshot(values={'route_selector': '', 'route_log': ['Start_node', 'Route_select_to_a', 'Route_select_to_b', 'End_node'], 'visit_count': 0}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f07cc63-0929-6518-8004-5fc3bf68ab9d'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2025-08-19T06:31:50.876689+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f07cc63-0928-662c-8003-a198172bbe26'}}, tasks=(), interrupts=())

In [25]:
from pydantic import BaseModel, ValidationError


class Model(BaseModel):
    x: list[int]


try:
    Model(x=1)
except ValidationError as exc:
    print(exc.errors()[0])
    print(repr(exc.errors()[0]["type"]))
    # > 'list_type'

{'type': 'list_type', 'loc': ('x',), 'msg': 'Input should be a valid list', 'input': 1, 'url': 'https://errors.pydantic.dev/2.11/v/list_type'}
'list_type'
